In [1]:
import sys
import os

# Add the project root to sys.path so we can import services
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)
    

from services.tts_service import tts_service

# Verify initialization
print(f"TTS Engine status: {'Available' if tts_service.tts else 'Not Initialized'}")

/mnt/data2/ntmduy/wiki-host/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
llama_context: n_ctx_per_seq (2048) < n_ctx_train (4096) -- the full capacity of the model will not be utilized
llama_context: n_ctx_per_seq (2048) < n_ctx_train (4096) -- the full capacity of the model will not be utilized


TTS Engine status: Available


In [4]:
output_path = tts_service.synthesize(
    text="Tham gia team dev trẻ trung, thoải mái và cực kỳ thân thiện tại SFIN",
    self_clone=True
)

In [5]:
output_path

'outputs/tts/tts_c5b0437c9ab3.wav'

## OmniVoice Vietnamese test

Model: `splendor1811/omnivoice-vietnamese`. Use a clean 3-5 second reference clip with an exact transcript for voice cloning.


In [11]:
import os
import sys

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = Path("outputs/tts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OMNIVOICE_MODEL_ID = "splendor1811/omnivoice-vietnamese"
OMNIVOICE_DEVICE = "cuda:1"  # change to cuda:0/cuda:2/etc. if needed
OMNIVOICE_SPEED = 0.9  # lower = slower, 1.0 = default, try 0.75-0.9
REF_AUDIO = DATA_DIR / "voice" / "ref_voice.MP3"
REF_TEXT = (DATA_DIR / "voice" / "ref_sub.txt").read_text(encoding="utf-8").strip()
TEST_TEXT = "Anh/chị có thể giới thiệu ngắn gọn về bản thân, tập trung vào hành trình từ khi học đại học đến hiện tại, và lý do nào khiến anh/chị chọn theo đuổi lĩnh vực Machine Learning / AI?"

print("ref_audio:", REF_AUDIO)
print("ref_text:", REF_TEXT)


ref_audio: /mnt/data2/ntmduy/wiki-host/data/voice/ref_voice.MP3
ref_text: Hồi mình còn học đại học, có những ngày trong túi chẳng có tiền tiêu, thứ duy nhất mà mình còn là xăng trong con xe CUP cũ.


In [8]:
import torch
from omnivoice import OmniVoice

omnivoice_model = OmniVoice.from_pretrained(
    OMNIVOICE_MODEL_ID,
    device_map=OMNIVOICE_DEVICE,
    dtype=torch.float16,
)
print("Loaded", OMNIVOICE_MODEL_ID, "on", OMNIVOICE_DEVICE)


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

Loaded splendor1811/omnivoice-vietnamese on cuda:1


In [9]:
omnivoice_prompt = omnivoice_model.create_voice_clone_prompt(
    ref_audio=str(REF_AUDIO),
    ref_text=REF_TEXT,
)
print("Voice prompt cached")


Voice prompt cached


In [12]:
import soundfile as sf

omnivoice_audio = omnivoice_model.generate(
    text=TEST_TEXT,
    language="vietnamese",
    voice_clone_prompt=omnivoice_prompt,
    speed=OMNIVOICE_SPEED,
)

speed_label = str(OMNIVOICE_SPEED).replace(".", "p")
omnivoice_output = OUTPUT_DIR / f"omnivoice_vietnamese_speed_{speed_label}.wav"
sf.write(str(omnivoice_output), omnivoice_audio[0], 24000)
omnivoice_output


PosixPath('outputs/tts/omnivoice_vietnamese_speed_0p9.wav')

### OmniVoice warmup note

Do not call `torch.compile(omnivoice_model)` or compile the full OmniVoice module in this environment. With the current `torch==2.5.1+cu121` + `transformers==5.8.1`, full-module compilation can make the internal Hugging Face LLM return an empty `ModelOutput`, which causes `IndexError: tuple index out of range` at `hidden_states = llm_outputs[0]`.

Use a normal inference warmup instead. Keep the model loaded and reuse `omnivoice_prompt` for speed.


In [ ]:
from omnivoice import OmniVoiceGenerationConfig
import torch

# Warmup without torch.compile. This preloads kernels/cache without breaking ModelOutput.
warmup_config = OmniVoiceGenerationConfig(num_step=8, guidance_scale=2.0)
with torch.inference_mode():
    _ = omnivoice_model.generate(
        text="Xin chào.",
        language="vietnamese",
        voice_clone_prompt=omnivoice_prompt,
        speed=OMNIVOICE_SPEED,
        generation_config=warmup_config,
    )
print("OmniVoice warmup complete")
